# 🎯 NRA Target Scorer — Cloud GPU Training & Active Learning Pipeline

Welcome to the official Google Colab environment for the **Camera-Independent NRA Pistol Target Scorer & Metrology Engine**!

### Capabilities Supported in this Notebook:
1. **High-Resolution Metrology:** Train U-Net models at $512 \times 512$, $800 \times 800$, or competition standard **$1024 \times 1024$** ($4.0\text{ px/mm}$, achieving **$0.335\text{ mm}$ sub-pixel precision**).
2. **Multi-Mode Training Strategies:**
   - **Synthetic Procedural:** 100% procedural cardboard, bullet tear, and handwriting synthesis.
   - **Option B (Pure Real Fine-Tuning):** Rapid domain adaptation using real target photographs (`real_ratio = 1.0`).
   - **Option A (Hybrid Blended Training):** Blends 70% synthetic edge cases with 30% real paper photographs (`real_ratio = 0.35`).
3. **Assisted Auto-Labeling (`target-scorer autolabel`):** Pre-rectify smartphone target photos, detect candidate bullet impacts, and render human-in-the-loop review cards with green crosshairs and score tags.
4. **Interactive In-Browser Visual Scorecards:** Inspect 10-shot NRA competition targets (NRA B-2 Slow Fire and NRA B-3 Timed/Rapid Fire) directly in Colab.
5. **One-Click Local Checkpoint Download:** Download trained weights directly to your local workstation.

---
### Step 1: Connect to GPU Runtime
In the top menu, go to **Runtime** > **Change runtime type**, select **T4 GPU** (or **A100** for fastest $1024 \times 1024$ convergence), and click **Save**.

In [ ]:
# Check GPU hardware and VRAM
!nvidia-smi
import torch
print("\n" + "=" * 50)
print("PyTorch Version:", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:     ", torch.cuda.get_device_name(0))
    print("VRAM Capacity:  ", f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 50)

## Step 2: Set Up Codebase

**Upload `target_scorer_colab_bundle.zip`:**
1. In the left sidebar, click the **Folder icon** 📁 to open files.
2. Drag and drop your local `target_scorer_colab_bundle.zip` (and optional pre-trained weights or real target zip).
3. Run the cell below to unpack and install the package in editable mode.

In [ ]:
import os
import shutil

# If bundle was uploaded:
if os.path.exists("/content/target_scorer_colab_bundle.zip"):
    print("[*] Unpacking target_scorer_colab_bundle.zip...")
    !unzip -q -o /content/target_scorer_colab_bundle.zip -d /content/
elif os.path.exists("/content/target_scorer"):
    print("[*] Found existing target_scorer directory.")
else:
    print("[!] Please drag and drop target_scorer_colab_bundle.zip into the Colab Files panel on the left!")

# Navigate into the project and install in editable mode
%cd /content/target_scorer
!pip install -q -e .
print("\n[✓] Target Scorer environment configured successfully!")

## Step 3: (Optional) Assisted Auto-Labeling for Real Target Datasets

If you uploaded new smartphone photographs of targets, run `target-scorer autolabel` to automatically pre-rectify the targets into $1024 \times 1024$ canonical space, detect bullet holes, score the shots, and output JSON annotations alongside visual human-in-the-loop review cards.

In [ ]:
# @title Assisted Auto-Labeling Tool { run: "auto" }
INPUT_DIR = "/content/target_scorer/datasets/calibration_targets/raw" # @param {type:"string"}
OUTPUT_DIR = "/content/target_scorer/datasets/autolabeled" # @param {type:"string"}
TARGET_TYPE = "B-2" # @param ["B-2", "B-3"]
CONF_THRESHOLD = 0.40 # @param {type:"number"}

if os.path.exists(INPUT_DIR):
    print(f"[*] Auto-labeling targets in '{INPUT_DIR}'...")
    !target-scorer autolabel \
        --input-dir {INPUT_DIR} \
        --output-dir {OUTPUT_DIR} \
        --target {TARGET_TYPE} \
        --threshold {CONF_THRESHOLD}
else:
    print(f"[*] Input directory '{INPUT_DIR}' not found. Skipping auto-labeling.")

## Step 4: Configure Training Parameters & Strategy

Choose between:
- **100% Synthetic Procedural:** Infinite algorithmic variation of cardboard, lighting, and bullet groupings.
- **Option B (Pure Real Fine-Tuning):** Rapidly adapt model weights to real paper fiber tears and range lighting (`real_ratio = 1.0`).
- **Option A (Hybrid Blended Training):** Combine 70% synthetic variations with 30% real target photos (`real_ratio = 0.35`).

In [ ]:
# @title Training Configuration { run: "auto" }
CANVAS_PRESET = "Ultra-Res (1024x1024 @ 4.0 px/mm)" # @param ["Ultra-Res (1024x1024 @ 4.0 px/mm)", "High-Res (800x800 @ 4.0 px/mm)", "Baseline (512x512 @ 4.0 px/mm)", "Quick Test (256x256 @ 2.0 px/mm)"]
TRAINING_MODE = "Option A: Hybrid (70% Syn + 30% Real)" # @param ["Synthetic Procedural (100% Syn)", "Option B: Pure Real Fine-Tuning (100% Real)", "Option A: Hybrid (70% Syn + 30% Real)"]
EPOCHS = 12 # @param {type:"integer"}
BATCH_SIZE = 8 # @param {type:"integer"}
LEARNING_RATE = 1.5e-4 # @param {type:"number"}
REAL_DATA_DIR = "/content/target_scorer/datasets/calibration_targets" # @param {type:"string"}
INITIAL_WEIGHTS = "/content/target_scorer/weights/target_scoring_hrnet_1024_best.pt" # @param {type:"string"}
USE_AMP = True # @param {type:"boolean"}

if "1024x1024" in CANVAS_PRESET:
    CANVAS_SIZE = 1024
    SCALE = 4.0
    SIGMA = 3.2
elif "800x800" in CANVAS_PRESET:
    CANVAS_SIZE = 800
    SCALE = 4.0
    SIGMA = 2.8
elif "512x512" in CANVAS_PRESET:
    CANVAS_SIZE = 512
    SCALE = 4.0
    SIGMA = 2.5
else:
    CANVAS_SIZE = 256
    SCALE = 2.0
    SIGMA = 1.8

if "Pure Real" in TRAINING_MODE:
    REAL_RATIO = 1.0
elif "Hybrid" in TRAINING_MODE:
    REAL_RATIO = 0.35
else:
    REAL_RATIO = 0.0

CHECKPOINT_PATH = f"/content/target_scorer/weights/target_scoring_hrnet_{CANVAS_SIZE}_best.pt"
os.makedirs("/content/target_scorer/weights", exist_ok=True)

print(f"[*] Canvas: {CANVAS_SIZE}x{CANVAS_SIZE} @ {SCALE} px/mm (Sigma: {SIGMA})")
print(f"[*] Mode:   {TRAINING_MODE} (Real Ratio: {REAL_RATIO})")
print(f"[*] Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"[*] Output Checkpoint: {CHECKPOINT_PATH}")

## Step 5: Execute Training on Cloud GPU

Run the cell below to execute training. Mixed precision (AMP) is enabled automatically on Volta/Turing/Ampere GPUs.

In [ ]:
amp_flag = "--amp" if USE_AMP else "--no-amp"
init_w_flag = f"--initial-weights {INITIAL_WEIGHTS} " if os.path.exists(INITIAL_WEIGHTS) else ""
real_flag = f"--real-data-dir {REAL_DATA_DIR} --real-ratio {REAL_RATIO} " if (REAL_RATIO > 0.0 and os.path.exists(REAL_DATA_DIR)) else ""

cmd = (
    f"target-scorer train "
    f"--canvas-size {CANVAS_SIZE} "
    f"--scale {SCALE} "
    f"--epochs {EPOCHS} "
    f"--batch-size {BATCH_SIZE} "
    f"--lr {LEARNING_RATE} "
    f"--sigma {SIGMA} "
    f"--checkpoint {CHECKPOINT_PATH} "
    f"{init_w_flag}"
    f"{real_flag}"
    f"{amp_flag}"
)

print(f"Executing: {cmd}\n")
!{cmd}


## Step 6: In-Browser Visual Inspection Scorecards

Evaluate the trained weights on both NRA B-2 and NRA B-3 official competition targets and render diagnostic scorecards right inside Colab:

In [ ]:
from IPython.display import Image, display

scorecard_b2 = f"/content/target_scorer/colab_scorecard_{CANVAS_SIZE}_b2.png"
scorecard_b3 = f"/content/target_scorer/colab_scorecard_{CANVAS_SIZE}_b3.png"

print("[*] Generating NRA B-2 Slow Fire Scorecard...")
!target-scorer infer \
    --checkpoint {CHECKPOINT_PATH} \
    --target B-2 \
    --canvas-size {CANVAS_SIZE} \
    --scale {SCALE} \
    --output {scorecard_b2}

print("\n[*] Generating NRA B-3 Timed/Rapid Fire Scorecard...")
!target-scorer infer \
    --checkpoint {CHECKPOINT_PATH} \
    --target B-3 \
    --canvas-size {CANVAS_SIZE} \
    --scale {SCALE} \
    --output {scorecard_b3}

if os.path.exists(scorecard_b2):
    print("\n[✓] Rendered NRA B-2 Visual Scorecard:")
    display(Image(scorecard_b2))

if os.path.exists(scorecard_b3):
    print("\n[✓] Rendered NRA B-3 Visual Scorecard:")
    display(Image(scorecard_b3))


## Step 7: Download Optimal Checkpoint to Your Workstation

Click below to download the trained checkpoint directly to your local computer:

In [ ]:
from google.colab import files
import os

if os.path.exists(CHECKPOINT_PATH):
    print(f"[*] Downloading '{CHECKPOINT_PATH}' to your computer...")
    files.download(CHECKPOINT_PATH)
else:
    print(f"[!] Checkpoint '{CHECKPOINT_PATH}' not found. Did training complete?")
